In [1]:
import time
import pubchempy as pcp
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import (
    BaggingRegressor,
    GradientBoostingRegressor,
    IsolationForest,
    RandomForestRegressor,
    StackingRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, plot_tree

RANDOM_STATE = 100926

def regression_metrics(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

def show_metrics(name, y_true, y_pred):
    values = regression_metrics(y_true, y_pred)
    print(name)
    print(f"  RMSE: {values['RMSE']:.3f}")
    print(f"  MAE:  {values['MAE']:.3f}")
    print(f"  R²:   {values['R2']:.3f}")
    return values

In [2]:
df_raw = pd.read_csv('antimicrobial_train_80pct.csv')

In [3]:
df_raw

,activity_id,molecule_chembl_id,pref_name,compound_name,canonical_smiles,standard_type,standard_units,assay_chembl_id,rdkit_MaxAbsEStateIndex,rdkit_MaxEStateIndex,...,rdkit_fr_sulfonamd,rdkit_fr_sulfone,rdkit_fr_term_acetylene,rdkit_fr_tetrazole,rdkit_fr_thiazole,rdkit_fr_thiocyan,rdkit_fr_thiophene,rdkit_fr_unbrch_alkane,rdkit_fr_urea,target_log10
0,946920,CHEMBL72286,NaN,NaN,O=C(c1ccccc1O)N1CCc2c(n(Cc3ccc(O)cc3)c3ccccc23)C1,MIC,ug.mL-1,CHEMBL677573,13.081525,13.081525,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.903090
1,379615,CHEMBL134905,NaN,NaN,CC(=O)NC[C@H]1CN(c2ccc3c(c2)CCCCN3C=O)C(=O)O1,MIC,ug.mL-1,CHEMBL677421,12.110715,12.110715,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.301030
2,16571259,CHEMBL3798190,NaN,NaN,CCn1cc(C(=O)O)c(=O)c2cc(F)c(N3CCNCC3)cc21.O=[N...,MIC,ug.mL-1,CHEMBL3801691,14.494909,14.494909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.892095
3,141787,CHEMBL320039,NaN,NaN,N[C@H]1CN(c2c(F)cc3c(=O)c(C(=O)O)cn(C4CC4)c3c2...,MIC,ug.mL-1,CHEMBL677359,15.030342,15.030342,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.221849
4,551390,CHEMBL88523,NaN,NaN,O=C(OCC1=CC=C(COC(=O)c2cccnc2)SS1)c1cccnc1,MIC,ug.mL-1,CHEMBL648210,11.880338,11.880338,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.301030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8470,1198295,CHEMBL1160682,NaN,NaN,C[C@@H](O)[C@H]1C(=O)N2C(C(=O)O)=C(Sc3nc(-c4cc...,MIC,ug.mL-1,CHEMBL673662,12.395635,12.395635,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.795880
8471,952203,CHEMBL88982,NaN,NaN,NCCNC(=O)c1cc(-c2c[nH]c3ccc(Br)cc23)nc2ccccc12...,MIC,ug.mL-1,CHEMBL673511,12.647314,12.647314,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.397940
8472,1190294,CHEMBL275692,NaN,NaN,CC(=O)OCC1=C(C(=O)OC(c2ccccc2)c2ccccc2)N2C(=O)...,MIC,ug.mL-1,CHEMBL670512,13.865431,13.865431,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.522879
8473,377863,CHEMBL44431,NaN,NaN,O=C(COc1ccccc1)N[C@@H]1C(=O)N2C(C(=O)O)=C(Cl)C...,MIC,ug.mL-1,CHEMBL673753,12.167056,12.167056,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.806180
